# Teste manual — Gold

Pré-requisitos:

- `pip install -e ".[dev]"` e `.env` configurado
- `python run_gold.py migrate`
- Silver/gold atualizados: `python run_sync.py daily --persist`

Fluxos:

- **Materializar:** `GoldOrchestrator` (silver → valor gold)
- **Ler SQLite:** `GoldReader` após `--persist`

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "src" / "app").is_dir():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src" / "app").is_dir():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Abra o notebook na raiz do repo ou em notebooks/")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from app.config import get_settings
from app.database import GoldReader
from app.lake.gold import BUILDER_NAMES, GoldOrchestrator

settings = get_settings()
orch = GoldOrchestrator()
reader = GoldReader()
START, END = "2026-01-01", "2026-01-31"

print("SILVER:", settings.silver_root)
print("DB:", settings.db_path)
print("Builders:", BUILDER_NAMES)

RuntimeError: Abra o notebook na raiz do repo ou em notebooks/

## 1. Feriados (materializer — SQL-ready)

Silver: `feriados/snapshot=1` (leitura integral, sem datas). Saída: `list[str]` ISO para tabela `FERIADOS`.

In [2]:
silver_feriados = orch.read_feriados()
display(silver_feriados["feriados"].head())

result_feriados = orch.materialize_feriados()
datas = result_feriados.value
print("total feriados:", len(datas))
print("primeiros:", datas[:5])
print("ultimos:", datas[-5:])

,data
0,2001-01-01
1,2001-02-26
2,2001-02-27
3,2001-04-13
4,2001-04-21


total feriados: 1263
primeiros: ['2001-01-01', '2001-02-26', '2001-02-27', '2001-04-13', '2001-04-21']
ultimos: ['2099-10-12', '2099-11-02', '2099-11-15', '2099-11-20', '2099-12-25']


In [3]:
silver_feriados

{'feriados':             data
 0     2001-01-01
 1     2001-02-26
 2     2001-02-27
 3     2001-04-13
 4     2001-04-21
 ...          ...
 1258  2099-10-12
 1259  2099-11-02
 1260  2099-11-15
 1261  2099-11-20
 1262  2099-12-25
 
 [1263 rows x 1 columns]}

## 2. CDI (materializer — SQL-ready)

Silver: partições `cdi/data=YYYY-MM-DD`. Passe a lista de datas (o pipeline define quais carregar).

In [2]:
dates = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_cdi = orch.materialize_cdi(dates)
display(result_cdi.value)
print("linhas:", len(result_cdi.value))

,data_referencia,cdi
0,2026-01-15,14.9
1,2026-01-16,14.9


linhas: 2


## 3. PTAX USD (materializer — SQL-ready)

Silver: partições `ptax/data=YYYY-MM-DD`. Passe a lista de datas (o pipeline define quais carregar).

In [2]:
dates_ptax = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_ptax = orch.materialize_ptax(dates_ptax)
display(result_ptax.value)
print("linhas:", len(result_ptax.value))

,data_referencia,ptax_compra,ptax_venda
0,2026-01-15,5.384,5.3846
1,2026-01-16,5.3792,5.3798


linhas: 2


## 4. BMF ajustes (materializer — SQL-ready)

Silver: partições `ajustes_bmf/data=YYYY-MM-DD`. Passe a lista de datas.

In [2]:
dates_bmf = ["2026-01-15", "2026-01-16"]  # exemplo; em produção o pipeline passa a lista
result_bmf = orch.materialize_bmf(dates_bmf)
display(result_bmf.value)
print("linhas:", len(result_bmf.value))

,data_referencia,data_vencimento,ticker,taxa_ajuste,quantidade_ajuste,codigo_isin
0,2026-01-15,2026-01-15,DAPF26,0.000,100000.00,BRBMEFDAP4M4
1,2026-01-15,2026-02-02,DI1G26,14.894,99341.04,BRBMEFD1I8F3
2,2026-01-15,2026-02-18,DAPG26,9.594,99203.40,BRBMEFDAP553
3,2026-01-15,2026-03-02,DI1H26,14.869,98363.28,BRBMEFD1I8G1
4,2026-01-15,2026-03-16,DAPH26,7.786,98816.93,BRBMEFDAP579
...,...,...,...,...,...,...
124,2026-01-16,2041-01-02,DI1F41,13.677,14881.38,BRBMEFD1I8T4
125,2026-01-16,2045-05-15,DAPK45,7.275,25963.05,BRBMEFDAP3B9
126,2026-01-16,2050-08-15,DAPQ50,7.210,18265.64,BRBMEFDAP3C7
127,2026-01-16,2055-05-17,DAPK55,7.155,13349.68,BRBMEFDAP3D5


linhas: 129


## 5. Mercado secundário (materializer — SQL-ready)

Silver: partições `mercado_secundario/data=YYYY-MM-DD`. Passe a lista de datas.

In [2]:
dates_ms = ["2026-01-15", "2026-01-16"]
result_ms = orch.materialize_mercado_secundario(dates_ms)
display(result_ms.value)
print("linhas:", len(result_ms.value))

,tipo_titulo,data_vencimento,data_referencia,taxa_anbima,intervalo_min_d0,intervalo_max_d0,intervalo_min_d1,intervalo_max_d1,pu,expressao,data_base,codigo_selic,codigo_isin,taxa_compra,taxa_venda,desvio_padrao,status
0,LFT,2026-03-01,2026-01-15,0.0221,-0.0474,0.0670,-0.0479,0.0673,18185.217583,Rentabilidade (% a.a.)/252,2000-07-01,210100,BRSTNCLF1RE0,0.0257,0.0203,0.000320,ATIVO
1,LTN,2026-04-01,2026-01-15,14.7330,14.6900,14.9439,14.6872,14.9385,972.038253,Taxa (% a.a.)/252,2024-01-05,100000,BRSTNCLTN8B5,14.7391,14.7253,0.000000,ATIVO
2,LTN,2026-07-01,2026-01-15,14.4124,14.2789,14.6536,14.2876,14.6586,941.412415,Taxa (% a.a.)/252,2023-01-06,100000,BRSTNCLTN848,14.4177,14.4070,0.002405,ATIVO
3,NTN-B,2026-08-15,2026-01-15,10.2999,9.8912,10.7313,9.9108,10.7557,4594.453474,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB4U6,10.3119,10.2862,0.001841,ATIVO
4,LFT,2026-09-01,2026-01-15,-0.0147,-0.0399,0.0556,-0.0400,0.0563,18187.363497,Rentabilidade (% a.a.)/252,2000-07-01,210100,BRSTNCLF1RF7,-0.0100,-0.0184,0.000226,ATIVO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,NTN-B,2040-08-15,2026-01-16,7.5204,7.3332,7.6765,7.3610,7.7045,4121.044008,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB3C6,7.5362,7.5085,0.001048,ATIVO
100,NTN-B,2045-05-15,2026-01-16,7.3854,7.2345,7.5641,7.2378,7.5676,4019.259772,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB0A6,7.4018,7.3705,0.005640,ATIVO
101,NTN-B,2050-08-15,2026-01-16,7.3229,7.1814,7.5035,7.1820,7.5044,4049.251613,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB3D4,7.3436,7.3084,0.003933,ATIVO
102,NTN-B,2055-05-15,2026-01-16,7.2700,7.1331,7.4537,7.1304,7.4514,3967.205126,Taxa (% a.a.)/252,2000-07-15,760199,BRSTNCNTB4Q4,7.2865,7.2535,0.000000,ATIVO


linhas: 104


data_referencia        str
cdi                float64
dtype: object

In [3]:
dates_liq = ["2026-01-15", "2026-01-16"]
result_liq = orch.materialize_liquidacoes_mercado(dates_liq)
display(result_liq.value)
print("linhas:", len(result_liq.value))

,tipo_titulo,data_vencimento,data_referencia,qtd_operacoes,qtd_titulos,pu_medio,status
0,LFT,2026-03-01,2026-01-15,29,255257,18185.235146,ATIVO
1,LTN,2026-04-01,2026-01-15,15,1080164,972.043333,ATIVO
2,LTN,2026-07-01,2026-01-15,16,940528,941.430919,ATIVO
3,NTN-B,2026-08-15,2026-01-15,74,479630,4595.095904,ATIVO
4,LFT,2026-09-01,2026-01-15,16,60132,18187.564957,ATIVO
...,...,...,...,...,...,...,...
92,NTN-B,2040-08-15,2026-01-16,1,1469,1672.063464,ATIVO
93,NTN-B,2045-05-15,2026-01-16,121,182658,4015.223357,ATIVO
94,NTN-B,2050-08-15,2026-01-16,2,1865,886.786272,ATIVO
95,NTN-B,2055-05-15,2026-01-16,1,134,665.222696,ATIVO


linhas: 97


In [6]:
from app.lake.gold import GoldOrchestrator

orch = GoldOrchestrator()
result = orch.materialize_leiloes(["2026-05-18", "2026-05-13"])
df = result.value  # DataFrame pronto para INSERT em LEILOES

In [7]:
df

,numero_edital,tipo_titulo,data_vencimento,data_referencia,oferta,quantidade_aceita,percentual_corte,oferta_segunda_volta,financeiro_aceito,financeiro_aceito_segunda_volta,quantidade_aceita_segunda_volta,pu_medio,taxa_media


In [12]:

orch = GoldOrchestrator()
result = orch.materialize_ipca_dict(["2026-04-25", "2026-04-27","2026-05-08","2026-05-13", "2026-05-14", "2026-05-15", "2026-05-22"])
df = result.value  # uma linha por data_referencia

In [13]:
df

,data_referencia,ultimo_mes_ipca,ref_month_atual,ref_month_anterior,indice_ipca_data_base,indice_ipca_fechado_atual,indice_ipca_fechado_anterior,var_ipca_atual,var_ipca_ant,ipca_proj,ipca_usado,usa_fechado,data_coleta_referencia,ipca_proj_data_coleta,inicio_mes_ipca,fim_mes_ipca
0,2026-04-25,3,2026-03-01,2026-03-01,1614.62,7545.53,7545.53,0.88,0.88,0.68,0.680000,0,2026-04-10,2026-04-10,2026-04-15,2026-05-15
1,2026-04-27,3,2026-03-01,2026-03-01,1614.62,7545.53,7545.53,0.88,0.88,0.68,0.680000,0,2026-04-10,2026-04-10,2026-04-15,2026-05-15
2,2026-05-08,3,2026-03-01,2026-03-01,1614.62,7545.53,7545.53,0.88,0.88,0.67,0.670000,0,2026-04-28,2026-04-28,2026-04-15,2026-05-15
3,2026-05-13,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.670066,1,2026-05-12,2026-05-12,2026-04-15,2026-05-15
4,2026-05-14,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.670066,1,2026-05-12,2026-05-12,2026-04-15,2026-05-15
5,2026-05-15,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
6,2026-05-22,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15


## 6. IPCA dict (protótipo do builder `ipca_dict`)

Silver:
- `ipca_indice` → índice e variação mensal fechada (SIDRA)
- `projecoes` → `variacao_projetada` ANBIMA (IPCA), com revisões por `data_coleta` / `data_validade`
- `feriados` → ajuste de dia 15 e dias úteis

Lógica espelhada do builder legado (`inicio_fim_mes_ipca`, `dicionario_ipca`): `IPCA_USADO` é fechado derivado do índice quando o último mês fechado é M−1 e a data está antes do dia 15; caso contrário usa projeção do mês IPCA em aberto (`fim_mes_ipca`).

In [3]:
from app.database import GoldReader
reader = GoldReader()
reader.ipca_dict.fetch_latest(20)

,data_referencia,ultimo_mes_ipca,ref_month_atual,ref_month_anterior,indice_ipca_data_base,indice_ipca_fechado_atual,indice_ipca_fechado_anterior,var_ipca_atual,var_ipca_ant,ipca_proj,ipca_usado,usa_fechado,data_coleta_referencia,ipca_proj_data_coleta,inicio_mes_ipca,fim_mes_ipca
0,2026-05-25,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
1,2026-05-24,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
2,2026-05-23,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
3,2026-05-22,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
4,2026-05-21,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
5,2026-05-20,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
6,2026-05-19,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
7,2026-05-18,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
8,2026-05-17,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15
9,2026-05-16,4,2026-04-01,2026-03-01,1614.62,7596.09,7545.53,0.67,0.88,0.50,0.500000,0,2026-05-12,2026-05-12,2026-05-15,2026-06-15


In [2]:
from app.lake.silver.reader import read_partition, list_partition_values

In [14]:
import pandas as pd


def _month_start(as_of: str | pd.Timestamp) -> str:
    ts = pd.Timestamp(as_of).normalize()
    return f"{ts.year:04d}-{ts.month:02d}-01"


def _ultimas_particoes_nao_vazias(
    dataset: str,
    as_of: str | pd.Timestamp,
    n: int = 2,
) -> pd.DataFrame:
    """Últimas ``n`` partições silver não vazias com reference_month <= mês de ``as_of``."""
    limite = _month_start(as_of)
    candidatos = sorted(
        (p for p in list_partition_values(dataset) if p <= limite),
        reverse=True,
    )
    frames: list[pd.DataFrame] = []
    for part in candidatos:
        if len(frames) >= n:
            break
        try:
            df = read_partition(dataset, part)
        except FileNotFoundError:
            continue
        if df is None or df.empty:
            continue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def silver_projecoes_e_ipca_ultimos_dois(
    as_of: str | pd.Timestamp,
    n=2
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Retorna (projecoes, ipca_indice): cada um é a união das 2 últimas
    partições mensais não vazias no silver até ``as_of``.
    """
    df_proj = _ultimas_particoes_nao_vazias("projecoes", as_of, n)
    df_ipca = _ultimas_particoes_nao_vazias("ipca_indice", as_of, n)
    return df_proj, df_ipca

In [15]:
import pandas as pd


def projecao_mais_recente(
    df_proj: pd.DataFrame,
    as_of: str | pd.Timestamp,
    *,
    indice: str | None = "IPCA",
    tipo_projecao: str | None = "PROJEÇÕES PARA O MÊS CORRENTE",
    ref_month: str | pd.Timestamp | None = None,
    respeitar_validade: bool = False,
) -> pd.Series:
    """
    Projeção com data_coleta <= as_of mais próxima de as_of (última coleta válida).

    Filtros opcionais: indice, tipo_projecao, ref_month (YYYY-MM-01).
    """
    as_of = pd.Timestamp(as_of).normalize()
    sub = df_proj.copy()

    if indice is not None:
        sub = sub[sub["indice"].astype(str).str.upper() == indice.upper()]
    if tipo_projecao is not None:
        sub = sub[sub["tipo_projecao"] == tipo_projecao]
    if ref_month is not None:
        mes = pd.Timestamp(ref_month).strftime("%Y-%m-%d")[:7] + "-01"
        sub = sub[sub["ref_month"].astype(str).str[:10] == mes]

    sub["data_coleta"] = pd.to_datetime(sub["data_coleta"], errors="coerce").dt.normalize()
    sub = sub[sub["data_coleta"].notna() & (sub["data_coleta"] <= as_of)]

    if respeitar_validade and "data_validade" in sub.columns:
        val = pd.to_datetime(sub["data_validade"], errors="coerce").dt.normalize()
        sub = sub[val.isna() | (val >= as_of)]

    if sub.empty:
        raise ValueError(f"Nenhuma projeção com data_coleta <= {as_of.date()}")

    # Mais próxima de as_of = maior data_coleta entre as <= as_of
    return sub.loc[sub["data_coleta"].idxmax()]


def projecao_mais_recente_valor(
    df_proj: pd.DataFrame,
    as_of: str | pd.Timestamp,
    **kwargs,
) -> tuple[float, pd.Timestamp]:
    row = projecao_mais_recente(df_proj, as_of, **kwargs)
    return (
        float(row["variacao_projetada"]),
        pd.Timestamp(row["data_coleta"]).normalize(),
    )
'''
def projecao_mais_recente_float(df_proj: pd.DataFrame, as_of: str | pd.Timestamp, **kwargs) -> float:
    return float(projecao_mais_recente(df_proj, as_of, **kwargs)["variacao_projetada"])
'''

'\ndef projecao_mais_recente_float(df_proj: pd.DataFrame, as_of: str | pd.Timestamp, **kwargs) -> float:\n    return float(projecao_mais_recente(df_proj, as_of, **kwargs)["variacao_projetada"])\n'

In [43]:
import pandas as pd
from app.lake.gold._io import read_silver_range
from app.lake.gold import GoldOrchestrator

# Constante do legado (NTN-B); futuramente pode vir de série histórica
INDICE_IPCA_DATA_BASE = 1614.62
'''
_TIPO_PROJ_CORRENTE = "PROJEÇÕES PARA O MÊS CORRENTE"
_TIPO_PROJ_POSTERIOR = "PROJEÇÕES PARA O MÊS POSTERIOR"
'''

def _feriados_set() -> set[str]:
    return set(GoldOrchestrator().materialize_feriados().value)


def e_dia_util(data: pd.Timestamp | str, feriados: set[str]) -> bool:
    ts = pd.Timestamp(data).normalize()
    if ts.weekday() >= 5:
        return False
    return ts.strftime("%Y-%m-%d") not in feriados


def adicionar_dias_uteis(
    data: pd.Timestamp | str,
    n_dias: int,
    feriados: set[str],
) -> pd.Timestamp:
    cur = pd.Timestamp(data).normalize()
    step = 1 if n_dias >= 0 else -1
    remaining = abs(int(n_dias))
    while remaining > 0:
        cur += pd.Timedelta(days=step)
        if e_dia_util(cur, feriados):
            remaining -= 1
    return cur.normalize()


def inicio_fim_mes_ipca(
    data: pd.Timestamp | str,
    feriados: set[str] | None = None,
) -> tuple[pd.Timestamp, pd.Timestamp]:
    feriados = feriados if feriados is not None else _feriados_set()
    data = pd.Timestamp(data).normalize()
    dia_15 = {
        "ant": (data - pd.DateOffset(months=1)).replace(day=15).normalize(),
        "atu": data.replace(day=15).normalize(),
        "prox": (data + pd.DateOffset(months=1)).replace(day=15).normalize(),
    }
    for key in dia_15:
        if not e_dia_util(dia_15[key], feriados):
            dia_15[key] = adicionar_dias_uteis(dia_15[key], 1, feriados)
    if data < dia_15["atu"]:
        return dia_15["ant"], dia_15["atu"]
    return dia_15["atu"], dia_15["prox"]


def _pick_ipca_month(
    monthly: pd.DataFrame,
    year: int,
    month: int,
    *,
    strict: bool = True,
) -> pd.Series:
    """Mês exato ou, se ``strict=False``, último ``ref_month`` publicado <= alvo."""
    alvo = pd.Timestamp(year=year, month=month, day=1)
    mask = (monthly["ref_month"].dt.year == year) & (monthly["ref_month"].dt.month == month)
    rows = monthly.loc[mask]
    if not rows.empty:
        return rows.iloc[-1]
    if strict:
        raise KeyError(f"ipca_indice sem ref_month {year:04d}-{month:02d}")
    prior = monthly[monthly["ref_month"] <= alvo]
    if prior.empty:
        raise KeyError(f"ipca_indice sem histórico em ou antes de {year:04d}-{month:02d}")
    return prior.iloc[-1]

def ipca_fechado_from_monthly(
    ipca_monthly: pd.DataFrame,
    as_of: pd.Timestamp | str,
    data_coleta: pd.Timestamp | str,
) -> dict[str, float | int]:
    """
    IPCA fechado (atual + anterior) conforme divulgação.

    ``data_coleta``: data em que o índice do mês M-1 (em relação a essa data) passa
    a estar disponível (ex. divulgação IBGE em 12/05 → índice de abril).

    Se ``as_of >= data_coleta`` → usa esse mês; senão → mês anterior.
    """
    as_of = pd.Timestamp(as_of).normalize()
    div = pd.Timestamp(data_coleta).normalize()

    mes_indice_na_div = div.replace(day=1) - pd.DateOffset(months=1)
    ultimo_ref = (
        mes_indice_na_div
        if as_of >= div
        else mes_indice_na_div - pd.DateOffset(months=1)
    )
    anterior_ref = ultimo_ref - pd.DateOffset(months=1)

    monthly = ipca_monthly.copy()
    monthly["ref_month"] = pd.to_datetime(monthly["ref_month"]).dt.normalize()
    monthly = monthly.sort_values("ref_month").reset_index(drop=True)

    def _pick(ref: pd.Timestamp) -> pd.Series:
        rows = monthly[monthly["ref_month"] == ref]
        if rows.empty:
            raise KeyError(f"ipca_indice sem ref_month {ref:%Y-%m-%d}")
        return rows.iloc[-1]

    row_atual = _pick(ultimo_ref)
    try:
        row_ant = _pick(anterior_ref)
    except KeyError:
        i = int(monthly.index[monthly["ref_month"] == ultimo_ref][0])
        row_ant = monthly.iloc[i - 1] if i > 0 else row_atual

    return {
        "ULTIMO_MES_IPCA": int(row_atual["ref_month"].month),
        "REF_MONTH_ATUAL": row_atual["ref_month"].strftime("%Y-%m-%d"),
        "REF_MONTH_ANTERIOR": row_ant["ref_month"].strftime("%Y-%m-%d"),
        "DATA_COLETA_REFERENCIA": div.strftime("%Y-%m-%d"),
        "INDICE_IPCA_FECHADO_ATUAL": float(row_atual["ipca_index"]),
        "INDICE_IPCA_FECHADO_ANTERIOR": float(row_ant["ipca_index"]),
        "VAR_IPCA_ATUAL": float(row_atual["ipca_mom"]),
        "VAR_IPCA_ANTERIOR": float(row_ant["ipca_mom"]),
    }

'''
def ipca_fechado_from_monthly(
    ipca_monthly: pd.DataFrame,
    as_of: pd.Timestamp | str,
) -> dict[str, float | int]:
    """Último mês fechado = M-1 da data; anterior = M-2 (equivalente ao iloc do SIDRA legado)."""
    as_of = pd.Timestamp(as_of).normalize()
    mes_atual = as_of - pd.DateOffset(months=1)
    mes_anterior = as_of - pd.DateOffset(months=2)

    monthly = ipca_monthly.copy()
    monthly["ref_month"] = pd.to_datetime(monthly["ref_month"])
    monthly = monthly.sort_values("ref_month").reset_index(drop=True)

    mes_limite = mes_atual.to_period("M").to_timestamp()
    cand = monthly[monthly["ref_month"] <= mes_limite]
    if cand.empty:
        cand = monthly[monthly["ref_month"] <= as_of.replace(day=1)]
    if cand.empty:
        cand = monthly
    row_atual = cand.iloc[-1]
    idx_atual = int(monthly.index[monthly["ref_month"] == row_atual["ref_month"]][-1])
    if idx_atual == 0:
        # Histórico curto no lake: ainda monta o dict; ``dicionario_ipca`` usa projeção.
        row_ant = row_atual
    else:
        row_ant = monthly.iloc[idx_atual - 1]

    return {
        "ULTIMO_MES_IPCA": int(row_atual["ref_month"].month),
        "INDICE_IPCA_FECHADO_ATUAL": float(row_atual["ipca_index"]),
        "INDICE_IPCA_FECHADO_ANTERIOR": float(row_ant["ipca_index"]),
        "VAR_IPCA_ATUAL": float(row_atual["ipca_mom"]),
        "VAR_IPCA_ANTERIOR": float(row_ant["ipca_mom"]),
    }
'''

def _projecoes_ipca_validas(
    projecoes: pd.DataFrame,
    as_of: pd.Timestamp,
) -> pd.DataFrame:
    sub = projecoes.loc[projecoes["indice"].astype(str).str.upper() == "IPCA"].copy()
    if sub.empty:
        raise ValueError("projecoes sem linhas IPCA")

    sub["ref_month"] = pd.to_datetime(sub["ref_month"], errors="coerce").dt.normalize()
    sub["data_coleta"] = pd.to_datetime(sub["data_coleta"], errors="coerce").dt.normalize()
    sub["data_validade"] = pd.to_datetime(sub["data_validade"], errors="coerce").dt.normalize()

    sub = sub[sub["ref_month"].notna() & sub["data_coleta"].notna()]
    sub = sub[sub["data_coleta"] <= as_of]
    sub = sub[sub["data_validade"].isna() | (sub["data_validade"] >= as_of)]
    if sub.empty:
        raise ValueError(f"projecoes sem IPCA válido em {as_of.date()}")
    return sub


def _pick_projecao_row(cand: pd.DataFrame) -> pd.Series:
    latest_coleta = cand["data_coleta"].max()
    sub = cand[cand["data_coleta"] == latest_coleta]
    for tipo in (_TIPO_PROJ_CORRENTE, _TIPO_PROJ_POSTERIOR):
        prefer = sub[sub["tipo_projecao"] == tipo]
        if not prefer.empty:
            return prefer.iloc[0]
    return sub.iloc[0]


def dicionario_ipca(
    as_of: pd.Timestamp | str,
    fechado: dict[str, float | int],
    ipca_proj_float: float,
    *,
    ipca_proj_data_coleta: pd.Timestamp | str | None = None,
    feriados: set[str] | None = None,
) -> dict[str, float | int]:
    """
    Monta o dict gold do IPCA (legado).
    ``usa_fechado`` quando o último mês fechado disponível é M-1 da data
    e a data está antes do dia 15 útil do mês corrente; senão usa projeção.
    """
    feriados = feriados if feriados is not None else _feriados_set()
    data = pd.Timestamp(as_of).normalize()
    dia_15_mes_atu = data.replace(day=15).normalize()
    if not e_dia_util(dia_15_mes_atu, feriados):
        dia_15_mes_atu = adicionar_dias_uteis(dia_15_mes_atu, 1, feriados)
    mes_m1 = (data - pd.DateOffset(months=1)).replace(day=1)
    ref_atual = pd.Timestamp(fechado["REF_MONTH_ATUAL"])
    usa_fechado = (
        ref_atual == mes_m1
        and data < dia_15_mes_atu
        and float(fechado["INDICE_IPCA_FECHADO_ATUAL"])
        != float(fechado["INDICE_IPCA_FECHADO_ANTERIOR"])
    )
    if usa_fechado:
        ipca_usado = (
            float(fechado["INDICE_IPCA_FECHADO_ATUAL"])
            / float(fechado["INDICE_IPCA_FECHADO_ANTERIOR"])
            - 1
        ) * 100
    else:
        ipca_usado = float(ipca_proj_float)
    out: dict[str, float | int | str] = {
        "ULTIMO_MES_IPCA": int(fechado["ULTIMO_MES_IPCA"]),
        "REF_MONTH_ATUAL": str(fechado["REF_MONTH_ATUAL"]),
        "REF_MONTH_ANTERIOR": str(fechado["REF_MONTH_ANTERIOR"]),
        "INDICE_IPCA_DATA_BASE": INDICE_IPCA_DATA_BASE,
        "INDICE_IPCA_FECHADO_ATUAL": float(fechado["INDICE_IPCA_FECHADO_ATUAL"]),
        "INDICE_IPCA_FECHADO_ANTERIOR": float(fechado["INDICE_IPCA_FECHADO_ANTERIOR"]),
        "VAR_IPCA_ATUAL": float(fechado["VAR_IPCA_ATUAL"]),
        "VAR_IPCA_ANTERIOR": float(fechado["VAR_IPCA_ANTERIOR"]),
        "IPCA_PROJ": float(ipca_proj_float),
        "IPCA_USADO": float(ipca_usado),
        "USA_FECHADO": usa_fechado,
        "DATA_COLETA_REFERENCIA": str(fechado["DATA_COLETA_REFERENCIA"]),
    }
    if ipca_proj_data_coleta is not None:
        out["IPCA_PROJ_DATA_COLETA"] = pd.Timestamp(ipca_proj_data_coleta).strftime(
            "%Y-%m-%d"
        )
    return out
def build_ipca_dict(
    as_of: pd.Timestamp | str,
    ipca_monthly: pd.DataFrame,
    projecoes: pd.DataFrame,
    feriados: set[str] | None = None,
) -> dict[str, float | int | str | bool]:
    feriados = feriados if feriados is not None else _feriados_set()
    data = pd.Timestamp(as_of).normalize()
    inicio, fim = inicio_fim_mes_ipca(data, feriados)
    ipca_proj, data_coleta_proj = projecao_mais_recente_valor(
        projecoes,
        data,
        respeitar_validade=False,
    )
    fechado = ipca_fechado_from_monthly(
        ipca_monthly,
        data,
        data_coleta=data_coleta_proj,
    )
    out = dicionario_ipca(
        data,
        fechado,
        ipca_proj,
        ipca_proj_data_coleta=data_coleta_proj,
        feriados=feriados,
    )
    out["INICIO_MES_IPCA"] = inicio.strftime("%Y-%m-%d")
    out["FIM_MES_IPCA"] = fim.strftime("%Y-%m-%d")
    return out

In [49]:
data = "2026-05-15"
df_proj = silver_projecoes_e_ipca_ultimos_dois(data, n=3)[0]
df_indice = silver_projecoes_e_ipca_ultimos_dois(data, n=3)[1]
proj = projecao_mais_recente_valor(df_proj, data)[0]
data_coleta = projecao_mais_recente_valor(df_proj, data)[1]
feriados = _feriados_set()
inicio_fim_mes_ipca(data)
ipca_fechado_from_monthly(
    ipca_monthly=df_indice,
    as_of=data,
    data_coleta=data_coleta
)
build_ipca_dict(as_of=data,
                 ipca_monthly=df_indice,
                 projecoes=df_proj,
                 feriados=feriados )



{'ULTIMO_MES_IPCA': 4,
 'REF_MONTH_ATUAL': '2026-04-01',
 'REF_MONTH_ANTERIOR': '2026-03-01',
 'INDICE_IPCA_DATA_BASE': 1614.62,
 'INDICE_IPCA_FECHADO_ATUAL': 7596.09,
 'INDICE_IPCA_FECHADO_ANTERIOR': 7545.53,
 'VAR_IPCA_ATUAL': 0.67,
 'VAR_IPCA_ANTERIOR': 0.88,
 'IPCA_PROJ': 0.5,
 'IPCA_USADO': 0.5,
 'USA_FECHADO': False,
 'DATA_COLETA_REFERENCIA': '2026-05-12',
 'IPCA_PROJ_DATA_COLETA': '2026-05-12',
 'INICIO_MES_IPCA': '2026-05-15',
 'FIM_MES_IPCA': '2026-06-15'}

In [ ]:
import brazilian_bonds_db as bbdb
from app.providers.uptodata import scrap_ajustes_bmf

d = "2026-05-25"  # sua data



raw = scrap_ajustes_bmf(d)
print("RAW DAP:", raw["TckrSymb"].str.startswith("DAP").sum())
print("RAW DI1:", raw["TckrSymb"].str.startswith("DI1").sum())

data = bbdb.read_data()  # seu data_root
gold = data.ajustes_bmf.fetch_on(d)
print("GOLD DAP:", gold["ticker"].str.startswith("DAP").sum())
print("GOLD DI1:", gold["ticker"].str.startswith("DI1").sum())


2026-05-26 10:00:24 - app.lake.bronze.pipeline - INFO - === Bronze phase (9 datasets) ===
2026-05-26 10:00:28 - app.lake.bronze.pipeline - INFO - [mercado_secundario] Bronze success: candidates=8 processed=['2026-05-25'] rows=51 path=\\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics\data\raw\mercado_secundario\data=2026-05-25\part.json
2026-05-26 10:00:36 - app.lake.bronze.pipeline - INFO - [liquidacoes_mercado] Bronze success: candidates=8 processed=['2026-05-25'] rows=57 path=\\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics\data\raw\liquidacoes_mercado\data=2026-05-25\part.parquet
2026-05-26 10:00:36 - app.providers.uptodata.client - INFO - Processing date: 2026-01-01
2026-05-26 10:00:36 - app.providers.uptodata.client - WARNING - Folder not found: X:\Interest_Rate\SettlementPrice\20260101\
2026-05-26 10:00:36 - app.providers.uptodata.client - WARNING - Skipping 2026-01-01 — file not found
2026-05-26 10:00:36 - app.provid

feriados: items=1263
  persisted_rows=1263
cdi: rows=1 columns=2
  range: 2026-05-26 .. 2026-05-26
  persisted_rows=1
bmf: rows=69 columns=6
  range: 2026-05-25 .. 2026-05-25
  persisted_rows=69
mercado_secundario: rows=51 columns=17
  range: 2026-05-25 .. 2026-05-25
  persisted_rows=51
liquidacoes_mercado: rows=51 columns=11
  range: 2026-05-25 .. 2026-05-25
  persisted_rows=51
ipca_dict: rows=1 columns=16
  range: 2026-05-26 .. 2026-05-26
  persisted_rows=1


2026-05-26 10:01:16 - app.providers.uptodata.client - INFO - Found 3 files with prefix Interest_Rate_SettlementPriceFile_Futures_20260525_
2026-05-26 10:01:16 - app.providers.uptodata.client - INFO - Using newest file: Interest_Rate_SettlementPriceFile_Futures_20260525_3.csv


RAW DAP: 22
RAW DI1: 47
GOLD DAP: 22
GOLD DI1: 47


In [5]:
from pathlib import Path

import pandas as pd

caminho = Path(
    r"Z:\Chila\projetos\brazil_fixed_income_analytics\data\silver\ajustes_bmf\data=2026-05-25\part.parquet"
)

df = pd.read_parquet(caminho)
df

,data_referencia,ticker,SctyId,SctySrc,MktIdrCd,codigo_isin,data_vencimento,quantidade_ajuste,taxa_ajuste,AdjstdQtStin,PrvsAdjstdQt,PrvsAdjstdQtTax,PrvsAdjstdQtStin,VartnPts,EqvtVal,AdjstdValCtrct,DataSts
0,2026-05-25,DAPF27,200001817282,8,BVMF,BRBMEFDAP4Y9,2027-01-15,94418.66,9.345,F,94414.91,9.352,U,3.75,NaN,7.131994,I
1,2026-05-25,DAPK27,100000164464,8,BVMF,BRBMEFDAP3U9,2027-05-17,92611.53,8.250,F,92608.81,8.253,U,2.72,NaN,5.173073,I
2,2026-05-25,DAPK29,200001361511,8,BVMF,BRBMEFDAP4N2,2029-05-15,80124.82,7.805,F,79959.43,7.881,U,165.39,NaN,314.549452,I
3,2026-05-25,DAPK31,300000055944,8,BVMF,BRBMEFDAP5E8,2031-05-15,68993.57,7.815,F,68819.17,7.870,U,174.40,NaN,331.685256,I
4,2026-05-25,DAPK33,100000191028,8,BVMF,BRBMEFDAP462,2033-05-16,59700.25,7.720,F,59488.54,7.775,U,211.71,NaN,402.643839,I
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,2026-05-25,DI1X29,100000250791,8,BVMF,BRBMEFD1I8Z1,2029-11-01,64368.88,13.745,F,64020.79,13.925,U,348.09,NaN,348.090000,N
65,2026-05-25,DI1Z26,400000105486,8,BVMF,BRBMEFD1I8O5,2026-12-01,93389.86,14.060,F,93366.09,14.116,U,23.77,NaN,23.770000,N
66,2026-05-25,DI1Z27,100000250768,8,BVMF,BRBMEFD1I8W8,2027-12-01,82334.15,13.720,F,82199.08,13.844,U,135.07,NaN,135.070000,N
67,2026-05-25,DI1Z28,100000250782,8,BVMF,BRBMEFD1I8Y4,2028-12-01,72522.45,13.667,F,72241.39,13.843,U,281.06,NaN,281.060000,N
